# Feature Engineering - Scaling, Normalization and Standardization

Well-designed Feature engineering is the process of creating, transforming or selecting important features from raw data to improve model performance. These features help the model capture useful patterns and relationships in the data.

## What is Feature Engineering?

Feature Engineering is the process of:

- Creating new features (columns)

- Transforming existing data

- Selecting useful variables

…so that Machine Learning models can learn patterns better and make more accurate predictions.

In simple terms:

`“Feature engineering means preparing and improving data so the model can understand it better.”`

It contributes to model building in the following ways:

- Well-designed features help the model to learn complex patterns more effectively.

- Removing noise and irrelevant information improves model prediction accuracy.

- Focusing on meaningful features helps the model to generalize better and reduces overfitting.

- Clear and informative features make the model easier to understand and interpret.

Why is Feature Engineering Important?

Feature engineering helps to:

- Improve model accuracy

- Reduce noise in data

- Handle missing values

- Convert raw data into meaningful information

- Make machine learning models smarter

Real-Life Example

Suppose you have this dataset:

| Name | Date of Birth | Salary |
| ---- | ------------- | ------ |
| John | 2000-05-10    | 300000 |
| Mary | 1998-08-20    | 450000 |

Instead of using Date of Birth directly, you can create:

| Age |
| --- |
| 25  |
| 27  |

This new column (Age) is more useful to a machine learning model.

That is Feature Engineering.

## Types of Feature Engineering

1. Handling Missing Values

Sometimes datasets contain empty values.

| Name  | Age |
| ----- | --- |
| John  | 25  |
| Mary  | NaN |
| David | 30  |

Methods for Handling Missing Values:

| Method            | Description                         |
| ----------------- | ----------------------------------- |
| Mean Imputation   | Replace missing values with average |
| Median Imputation | Replace with middle value           |
| Mode Imputation   | Replace with most frequent value    |
| Drop Rows         | Remove rows with missing values     |
| Forward Fill      | Use previous value                  |

Example

If ages are:

25, 30, 35, NaN

Mean = 30

Replace NaN with 30.

2. Encoding Categorical Variables

Machine learning models understand numbers, not text.

Example Dataset

| Gender |
| ------ |
| Male   |
| Female |
| Male   |

`We convert text into numbers.`

Types of Encoding

Label Encoding

| Gender | Encoded |
| ------ | ------- |
| Male   | 1       |
| Female | 0       |


One-Hot Encoding

| Male | Female |
| ---- | ------ |
| 1    | 0      |
| 0    | 1      |
| 1    | 0      |

Dataset overview - Housing.csv

Rows

1,460

Numeric columns

2

LotArea range

1,300 – 215,245

MSSubClass range

20 - 190

LotArea

Lot size in square feet. Large range (1,300–215,245) with a right-skewed distribution. Mean ≈ 10,517, std ≈ 9,981.

MSSubClass

Dwelling class code (categorical-ish integers). Values like 20, 30, 60, 70, 190. Mean ≈ 57, std ≈ 42.

Why scaling matters here: LotArea values are in the tens of thousands, while MSSubClass is under 200. Without scaling, any distance-based algorithm (KNN, SVM, K-Means) or gradient-based model would be dominated by LotArea, effectively ignoring MSSubClass.

In [56]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Datasets/Housing.csv')

df = df.select_dtypes(include=np.number)
df.head()

,LotArea,MSSubClass
0,8450,60
1,9600,20
2,11250,60
3,9550,70
4,14260,60


Line 1 — import pandas as pd
Imports the pandas library, aliased as pd. Pandas provides the DataFrame — the core structure for tabular data in Python. The alias pd is a universal convention.

Line 2 — import numpy as np
Imports NumPy, aliased as np. NumPy provides np.number — a dtype selector we'll use on line 5 — plus fast array math needed for scaling formulas.

Line 4 — pd.read_csv('Housing.csv')
Reads the CSV file into a DataFrame. The result has 1,460 rows × 2 columns. Pandas auto-detects column names from the header row.

Line 5 — select_dtypes(include=np.number) ⬅ key step
Filters the DataFrame to keep only numeric columns (int64, float64). This is critical before scaling — you cannot scale text columns like "Neighborhood" or "HouseStyle". np.number matches all numeric dtypes in one shot. Result: only LotArea and MSSubClass remain.

NOTE: This is the key preparation step. It drops any non-numeric columns (like Neighborhood, HouseStyle, etc. if they existed) because you cannot mathematically scale text. np.number catches all integer and float types at once. The result here keeps only LotArea and MSSubClass.

Line 6 — df.head()
Displays the first 5 rows as a quick sanity check. Always run .head() after loading or transforming data to verify the shape and values look right before scaling.

# Step 2: 

## 1.Apply Absolute Maximum Scaling

In [57]:
max_abs = np.max(np.abs(df), axis=0) # Calculates the maximum absolute value for each column.

scaled_df = df / max_abs # Divides each value by the maximum absolute value of its column to scale the data.

scaled_df.head()  # Displays the first few rows of the scaled dataset.

,LotArea,MSSubClass
0,0.039258,0.315789
1,0.044600,0.105263
2,0.052266,0.315789
3,0.044368,0.368421
4,0.066250,0.315789


## 2.Min-Max Scaling

Min-Max Scaling rescales features by subtracting the minimum value and dividing by the difference between the maximum and minimum values. This usually maps feature values to the range 0 to 1 while preserving the original distribution.

Implementation

- MinMaxScaler(): Creates a scaler object for Min-Max scaling.

- scaler.fit_transform(df): Calculates min and max values and scales the dataset between 0 and 1.

In [58]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)
scaled_df = pd.DataFrame(scaled_data, columns=df.columns)

scaled_df.head()

,LotArea,MSSubClass
0,0.033420,0.235294
1,0.038795,0.000000
2,0.046507,0.235294
3,0.038561,0.294118
4,0.060576,0.235294


### Line by line:

- MinMaxScaler() — instantiates the scaler object. No data is touched yet; it is just a configured tool. Default range is [0, 1] but you can override it with feature_range=(−1, 1) if needed.

- scaler.fit_transform(df) — the most important line. Under the hood, fit() scans every column to record its min and max values (1,300 and 215,245 for LotArea; 20 and 190 for MSSubClass). Then transform() applies the formula to every cell. The output is a raw NumPy array — column names are lost at this point.

- pd.DataFrame(scaled_data, columns=df.columns) — reconstructs the DataFrame, restoring column names from the original df. Without this, downstream code like scaled_df['LotArea'] would fail.

- scaled_df.head() — verifies the result. Every value should be in [0, 1]. Row 1's MSSubClass shows exactly 0.0 because its raw value is 20, which is the column minimum.

### Summary of what was done

The aim of this code is to rescale all numeric columns in the DataFrame to a range of [0, 1] so that no single feature dominates others due to differences in magnitude.

It does this in four steps:

- Creates a Min-Max scaler object

- Learns the min and max of each column and scales the data

- Converts the result back into a named DataFrame

- Previews the first 5 rows to confirm the output

In [59]:
print("Before Min-Max Scaling:")
print(df.head())

print("\nAfter Min-Max Scaling:")
print(scaled_df.head())

Before Min-Max Scaling:
   LotArea  MSSubClass
0     8450          60
1     9600          20
2    11250          60
3     9550          70
4    14260          60

After Min-Max Scaling:
    LotArea  MSSubClass
0  0.033420    0.235294
1  0.038795    0.000000
2  0.046507    0.235294
3  0.038561    0.294118
4  0.060576    0.235294


- Before scaling (df.head()) — raw values with very different magnitudes. LotArea is in the thousands while MSSubClass is in the tens/hundreds.

- After scaling (scaled_df.head()) — all values now sit between 0 and 1. The relative differences between rows are perfectly preserved, only the scale has changed.

Verifying Row 0 manually:

- For LotArea: (8450 − 1300) / (215245 − 1300) = 7150 / 213945 = 0.039258

- For MSSubClass: (60 − 20) / (190 − 20) = 40 / 170 = 0.315789

3. Normalization (Vector Normalization)

Normalization scales each data sample so that its vector length (Euclidean norm) becomes 1. It focuses on the direction of data points rather than their magnitude, making it useful in tasks like text classification and clustering.

Implementation

- Normalizer(): Creates a normalizer object to scale data.

- scaler.fit_transform(df): Normalizes each row so its vector length becomes 1.

In [60]:
from sklearn.preprocessing import Normalizer

scaler = Normalizer()
scaled_data = scaler.fit_transform(df)
scaled_df = pd.DataFrame(scaled_data, columns=df.columns)

scaled_df.head()

,LotArea,MSSubClass
0,0.999975,0.007100
1,0.999998,0.002083
2,0.999986,0.005333
3,0.999973,0.007330
4,0.999991,0.004208


### Explanation on the steps

- Step 1 — compute the Euclidean norm ‖X‖

‖X‖ = √(8450² + 60²)
     = √(71,402,500 + 3,600)
     = √71,406,100
     = 8,450.213

- Step 2 — divide each value by the norm

LotArea_scaled   = 8,450 / 8,450.213 = 0.999975

MSSubClass_scaled =     60 / 8,450.213 = 0.007100

- Step 3 — verify: scaled row vector length = 1

√(0.999975² + 0.007100²) = √(0.999950 + 0.000050) = √1.000000 = 1.0 

In [61]:
print("Before Vector Normalization:")
print(df.head())

print("\nAfter Vector Normalization:")
print(scaled_df.head())

Before Vector Normalization:
   LotArea  MSSubClass
0     8450          60
1     9600          20
2    11250          60
3     9550          70
4    14260          60

After Vector Normalization:
    LotArea  MSSubClass
0  0.999975    0.007100
1  0.999998    0.002083
2  0.999986    0.005333
3  0.999973    0.007330
4  0.999991    0.004208


## 4. Standardization

Standardization scales features by subtracting the mean and dividing by the standard deviation. This transforms the data so that features have zero mean and unit variance, which helps many machine learning models perform better.

Implementation

standardScaler(): Creates a scaler for standardizing the data.

scaler.fit_transform(df): Subtracts the mean and divides by the standard deviation.

In [62]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)
scaled_df = pd.DataFrame(scaled_data,
                         columns=df.columns)
print(scaled_df.head())

    LotArea  MSSubClass
0 -0.207142    0.073375
1 -0.091886   -0.872563
2  0.073480    0.073375
3 -0.096897    0.309859
4  0.375148    0.073375


The formula

`X_scaled = (X_i − μ) / σ `   (Z-score)

The numerator shifts the entire distribution so the mean lands at 0. Values above the mean become positive; values below become negative. The denominator stretches or compresses the spread so that one standard deviation equals 1. The result is called a Z-score — it tells you how many standard deviations a value is from the mean.

Here is the complete breakdown:

The formula — (Xi − μ) / σ. Subtracting the mean centers the data at zero. Dividing by the standard deviation makes one unit of spread equal exactly 1. The result is a Z-score — a universal measure of how far a value sits from its column's average.
Line by line:

StandardScaler() — creates the scaler. Unlike Normalizer, this one does learn from data. After fitting you can inspect what it learned via scaler.mean_ and scaler.scale_.

scaler.fit_transform(df) — fit() computes and stores the mean and standard deviation per column. transform() then applies the Z-score formula to every value. As always, in a real ML pipeline use fit_transform() only on training data and transform() alone on test data.

pd.DataFrame(scaled_data, columns=df.columns) — rebuilds the named DataFrame from the NumPy array output.

print(scaled_df.head()) — note this uses print() explicitly, unlike the previous scalers that just called scaled_df.head(). Both produce the same output in Jupyter, but print() is more portable across environments.

Reading the result — negative values mean that row is below the column mean; positive values mean above. Row 1's MSSubClass of −0.876 tells you that house's dwelling class is nearly one full standard deviation below average. There is no fixed range — standardized values are unbounded, which is the key difference from Min-Max.

## 5. Robust Scaling

Robust Scaling scales features using the median and interquartile range (IQR) instead of the mean and standard deviation. This makes it less sensitive to outliers and skewed data, making it suitable for datasets with extreme values or noise.

In [63]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
scaled_data = scaler.fit_transform(df)
scaled_df = pd.DataFrame(scaled_data,
                         columns=df.columns)
print(scaled_df.head())

    LotArea  MSSubClass
0 -0.254076         0.2
1  0.030015        -0.6
2  0.437624         0.2
3  0.017663         0.4
4  1.181201         0.2


Instead of using the mean and standard deviation (which both get pulled by extreme values), Robust Scaling uses the median as the center and the IQR as the spread. Since both the median and IQR are calculated from the middle 50% of the data, outliers at the extremes have virtually no effect on the scaling.

### Why median and IQR resist outliers

The problem with mean and std: LotArea has a max of 215,245 while most lots are 8,000–15,000 sqft. That one extreme value pulls the mean up from ~9,478 (median) to 10,517 and inflates the std to 9,981. The median and IQR ignore it completely — they only look at the middle 50% of values.

Here is the complete breakdown:

The formula — `(Xi − median) / IQR`. 

The median replaces the mean as the center point. The IQR (Q75 − Q25) replaces the standard deviation as the measure of spread. Both are calculated purely from the middle 50% of the data, so extreme values at the top or bottom have zero influence.

Line by line:

- nRobustScaler() — creates the scaler using the default 25th–75th percentile range. After fitting you can check what it learned via scaler.center_ for the medians and scaler.scale_ for the IQR values.

- scaler.fit_transform(df) — fit() computes median and IQR per column. For LotArea that gives median = 9,478 and IQR = 4,048. For MSSubClass, median = 50 and IQR = 50. transform() then applies the formula.

- pd.DataFrame + print() — same pattern as StandardScaler. Output is unbounded — values beyond ±1 simply mean that point is more than one IQR away from the median.

- Reading the result — Row 0's LotArea of −0.254 means it sits about a quarter of an IQR below the median. Row 4's +1.181 means it is 1.18 IQRs above. The output looks similar to Standardization but the crucial difference shows up at the extremes — the majority of typical houses stay clustered tightly around 0, undistorted by the outlier at 215,245.